<a href="https://colab.research.google.com/github/franciscogarate/mcaf/blob/master/notebooks/Ejemplo_concentracion_200m_Madrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculo del punto central con la mayor concentración geográfica

In [ ]:
!git clone https://github.com/franciscogarate/mcaf

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

Cargamos los datos:

In [ ]:
fichero = '212827-0-administracion-justicia-csv.csv'
#fichero = '201544-3-centros-salud-csv.csv'

In [ ]:
df = pd.read_csv(f'mcaf/data/{fichero}', sep=';', encoding='latin-1', usecols=['NOMBRE','COORDENADA-X','LATITUD','LONGITUD'])
df.dropna(inplace=True)
df = df.drop_duplicates(subset='COORDENADA-X', keep='first')
df['SUMA_ASEG'] = 1000
df.head()

Primero, centro el mapa en Madrid

In [ ]:
from folium import Map, Marker, plugins
from folium.plugins import HeatMap
map_madrid = Map(location=[40.42, -3.70], zoom_start = 12)

Creamos una matriz de coordenadas (lat, lon):

In [ ]:
coords = df[['LATITUD', 'LONGITUD']].to_numpy()

Muestro el mapa de calor

In [ ]:
coord_riesgos = np.asarray(coords)
HeatMap(coord_riesgos, radius=20).add_to(map_madrid)
map_madrid

Calculamos la matriz de distancias en grados

In [ ]:
distancias = cdist(coords, coords, metric='euclidean')
distancias

Sumamos las viviendas de todos los distritos a menos de 500 metros

In [ ]:
matrix_cumulos_500m = distancias < 200 / 100000  # Aproximación de 200 metros a 0.005 grados
matrix_cumulos_500m

In [ ]:
matrix_capitales = matrix_cumulos_500m * df['SUMA_ASEG'].values[np.newaxis, :]
matrix_capitales

In [ ]:
capital_concentrado = matrix_capitales.sum(axis=1)

Encontramos el índice del punto con mayor concentración

In [ ]:
indice_max = np.argmax(capital_concentrado)
punto_central = df.iloc[indice_max]
punto_central

Coordenadas del punto más denso en contador de viviendas

In [ ]:
print(f'Latitud: {punto_central['LATITUD']}, Longitud: {punto_central['LONGITUD']}')
print(f'Capital total radio 200 m: {capital_concentrado[indice_max]:,.0f}')
print(f'Número de viviendas del cúmulo: {matrix_cumulos_500m[indice_max].sum()}')

In [ ]:
Marker(location=[punto_central['LATITUD'], punto_central['LONGITUD']]).add_to(map_madrid)
map_madrid